In [2]:
import sys, os
import xarray as xr
import numpy as np

from pathlib import Path

# always stable relative to cwd in PBS jobs
ROOT = "/home/563/fm6730/localrepo/GC26-combined-solar-wind"

sys.path.insert(0, ROOT + "/src")

import analysis_func as af


In [3]:
sys.path

['/home/563/fm6730/localrepo/GC26-combined-solar-wind/src',
 '/home/563/fm6730/home',
 '/home/563/fm6730/ondemand/data/sys/dashboard/batch_connect/sys/jupyter/ncigadi/output/08a81bc1-3b19-4806-82c3-aedb86ce74f8/lib/python3',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python312.zip',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12/lib-dynload',
 '',
 '/g/data/xp65/public/apps/med_conda/envs/analysis3-26.05/lib/python3.12/site-packages']

In [4]:
from pathlib import Path

solar_cf = Path("/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/")

# List files
print(list(solar_cf.iterdir()))

# Example file
# file = solar_cf / "my_file.nc"

# print(file.exists())

[PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1980_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_2001_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1949_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1981_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1985_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1989_Aus.nc'), PosixPath('/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/raw/solar_cf/solar_capacity_factor_van_der_Wiel_era5_hourly_1950_Aus.nc')

In [5]:
year = 2003

In [6]:
folder_main = '/g/data/w42/dr6273/work/projects/Aus_energy/production_metrics'

In [7]:
folder_solar = f'{folder_main}/solar/capacity_factor/van_der_Wiel' 
folder_wind  = f'{folder_main}/wind/capacity_factor/van_der_Wiel' 

In [8]:
file_solar   = f'{folder_solar}/solar_capacity_factor_van_der_Wiel_era5_hourly_{year}_Aus.nc'
file_wind    = f'{folder_wind}/wind_capacity_factor_van_der_Wiel_era5_hourly_{year}_Aus.nc'

In [9]:
ds_solar = xr.open_dataset(file_solar)
ds_wind  = xr.open_dataset(file_wind)

In [10]:
da_solar = ds_solar['capacity_factor']
da_wind  = ds_wind['capacity_factor']

In [11]:
time = da_solar.time
lat = da_solar.lat
lon = da_solar.lon


In [12]:
daytime_mask = af.daytime_mask_xr(da_solar, "time", "lat", "lon")

In [13]:
da_solar_masked = xr.where(daytime_mask, da_solar, np.nan)

In [14]:
da_wind_masked = xr.where(daytime_mask, da_wind, np.nan)

In [15]:
da_wind_masked = da_wind_masked.rename('wind_cf').dropna(dim='time')

In [16]:
da_solar_masked = da_solar_masked.rename('solar_cf').dropna(dim='time')

In [17]:
da_solar_sel = xr.where(da_solar_masked > 0.1, 1, 0) # 1 is higher than 0.1
da_wind_sel  = xr.where(da_wind_masked > 0.1, 1, 0) # 1 is higher than 0.1

In [18]:
da_combined = da_solar_sel + da_wind_sel

In [19]:
da_combined = xr.where(da_combined == 0, 1, 0) # 1 is combined solar & wind lulls

In [20]:
da_combined

<xarray.DataArray (time: 2896, lat: 141, lon: 181)> Size: 591MB
array([[[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
...
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(2896, 141, 181))
Coordinates:
  * time     (time) datetime64[ns] 23kB 2003-01-01 ... 2003-12-31T23:00:00
  * lat      (lat) float32 564B -10.0 -10.25 -10.5 -10.75 ... -44.5 -44.75 -45.0
  * lon      (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0

In [45]:
da_sum_combined = da_combined.sum(dim='time')

In [46]:
import pandas as pd

In [47]:
da_sum_combined_wcoords = da_sum_combined.expand_dims(year=[int(year)]).assign_coords(year=[int(year)])

In [49]:

#Outputting

ds_out = xr.Dataset({
    'count_hour': da_sum_combined_wcoords,
    'percentage': da_sum_combined_wcoords/len(da_combined.time)*100.
    })

In [50]:
ds_out

<xarray.Dataset> Size: 410kB
Dimensions:     (lat: 141, lon: 181, year: 1)
Coordinates:
  * lat         (lat) float32 564B -10.0 -10.25 -10.5 ... -44.5 -44.75 -45.0
  * lon         (lon) float32 724B 110.0 110.2 110.5 110.8 ... 154.5 154.8 155.0
  * year        (year) int64 8B 2003
Data variables:
    count_hour  (year, lat, lon) int64 204kB 132 128 128 126 126 ... 20 20 23 25
    percentage  (year, lat, lon) float64 204kB 4.558 4.42 4.42 ... 0.7942 0.8633

In [54]:

folder_out   = Path("/home/563/fm6730/localrepo/GC26-combined-solar-wind/data/processed/daytime/")
os.makedirs(folder_out, exist_ok=True)

In [56]:

ds_out.to_netcdf(f"{folder_out}/daytime_days_capacity_factor_lower_than_10pc_{year}.nc")